In [2]:
import numpy as np
import pandas as pd

# groupby operation

## groupby columns

In [5]:
rng=np.random.default_rng(234321)
df=pd.DataFrame({'key1':['a','a',None,'b','b','a',None],
'key2':pd.Series([1,2,1,2,1,None,1],dtype='Int64'),
'data1':rng.standard_normal(7),
'data2':rng.standard_normal(7)})
df

,key1,key2,data1,data2
0,a,1,0.984036,-0.111297
1,a,2,-2.377249,-1.681970
2,None,1,-0.214540,-0.971601
3,b,2,-2.006021,-0.482860
4,b,1,0.678737,2.067048
5,a,<NA>,-0.690115,-2.102542
6,None,1,-0.091901,1.064763


In [8]:
group=df.groupby('key1')['data1']
group.mean()

key1
a   -0.694443
b   -0.663642
Name: data1, dtype: float64

In [10]:
ans=df['data1'].groupby([df['key1'],df['key2']]).mean()
ans

key1  key2
a     1       0.984036
      2      -2.377249
b     1       0.678737
      2      -2.006021
Name: data1, dtype: float64

In [11]:
ans.unstack()

key2,1,2
key1,,
a,0.984036,-2.377249
b,0.678737,-2.006021


In [12]:
df.groupby([df['key1'],df['key2']]).size()

key1  key2
a     1       1
      2       1
b     1       1
      2       1
dtype: int64

In [26]:
df.groupby([df['key1'],df['key2']],dropna=False).size()

key1  key2
a     1       1
      2       1
      <NA>    1
b     1       1
      2       1
NaN   1       2
dtype: int64

In [13]:
df.groupby([df['key1'],df['key2']]).count()

data1  data2
key1 key2              
a    1         1      1
     2         1      1
b    1         1      1
     2         1      1

**count()** calculate the number of non-missing value

**size()** calculate all of element

In [23]:
for group, value in df.groupby('key1'):
    print(group)
    print('----------------------------------------')
    print(f'{value}\n')

a
----------------------------------------
  key1  key2     data1     data2
0    a     1  0.984036 -0.111297
1    a     2 -2.377249 -1.681970
5    a  <NA> -0.690115 -2.102542

b
----------------------------------------
  key1  key2     data1     data2
3    b     2 -2.006021 -0.482860
4    b     1  0.678737  2.067048



In [25]:
pieces={name:group for name, group in df.groupby('key1')}
pieces['a']

,key1,key2,data1,data2
0,a,1,0.984036,-0.111297
1,a,2,-2.377249,-1.681970
5,a,<NA>,-0.690115,-2.102542


## groupby dictionary

In [27]:
rng = np.random.default_rng(seed=12345)
people = pd.DataFrame(rng.standard_normal((5, 5)),
                      columns=["a", "b", "c", "d", "e"],
                      index=["Joe", "Steve", "Wanda", "Jill", "Trey"])
people.iloc[(4, 1), (0, 3)] = np.nan
people.loc["Wanda", ["b", "c"]] = np.nan

people

,a,b,c,d,e
Joe,-1.423825,1.263728,-0.870662,-0.259173,-0.075343
Steve,-0.740885,-1.367793,0.648893,NaN,-1.952863
Wanda,2.347410,NaN,NaN,0.902198,-0.466953
Jill,-0.060690,0.788844,-1.256668,0.575858,1.398979
Trey,NaN,-0.299699,0.902919,-1.621583,-0.158189


In [28]:
mapping = {"a": "red", "b": "red", "c": "blue",
           "d": "blue", "e": "red", "f" : "orange", "g" : "violet"}
mapping

{'a': 'red',
 'b': 'red',
 'c': 'blue',
 'd': 'blue',
 'e': 'red',
 'f': 'orange',
 'g': 'violet'}

In [29]:
people.T.groupby(mapping).sum().T

,blue,red
Joe,-1.129835,-0.235440
Steve,0.648893,-4.061540
Wanda,0.902198,1.880456
Jill,-0.680811,2.127134
Trey,-0.718663,-0.457888


In [30]:
rng = np.random.default_rng(seed=12345)
people = people.assign(f = rng.standard_normal((5, 1)),
                       g = rng.standard_normal((5, 1)))
people

,a,b,c,d,e,f,g
Joe,-1.423825,1.263728,-0.870662,-0.259173,-0.075343,-1.423825,-0.740885
Steve,-0.740885,-1.367793,0.648893,NaN,-1.952863,1.263728,-1.367793
Wanda,2.347410,NaN,NaN,0.902198,-0.466953,-0.870662,0.648893
Jill,-0.060690,0.788844,-1.256668,0.575858,1.398979,-0.259173,0.361058
Trey,NaN,-0.299699,0.902919,-1.621583,-0.158189,-0.075343,-1.952863


In [31]:
people.T.groupby(mapping).sum().T

,blue,orange,red,violet
Joe,-1.129835,-1.423825,-0.235440,-0.740885
Steve,0.648893,1.263728,-4.061540,-1.367793
Wanda,0.902198,-0.870662,1.880456,0.648893
Jill,-0.680811,-0.259173,2.127134,0.361058
Trey,-0.718663,-0.075343,-0.457888,-1.952863


## groupby function

In [32]:
people.groupby(len).sum()

,a,b,c,d,e,f,g
3,-1.423825,1.263728,-0.870662,-0.259173,-0.075343,-1.423825,-0.740885
4,-0.060690,0.489146,-0.353749,-1.045725,1.240790,-0.334517,-1.591805
5,1.606525,-1.367793,0.648893,0.902198,-2.419816,0.393067,-0.718900


# data aggregation

In [35]:
# select n of the smallest in each group
grouped=df.groupby('key1')
grouped['data1'].nsmallest(1)

key1   
a     1   -2.377249
b     3   -2.006021
Name: data1, dtype: float64

In [36]:
def ran(df):
    return df.max()-df.min()

In [37]:
# agg(): apply the function to dataframe
grouped.agg(ran)

,key2,data1,data2
key1,,,
a,1,3.361285,1.991245
b,1,2.684758,2.549908


In [38]:
# describe(): display mutiple statistic measure in each columns
grouped.describe()

key2                                           data1            ...  \
     count mean       std  min   25%  50%   75%  max count      mean  ...   
key1                                                                  ...   
a      2.0  1.5  0.707107  1.0  1.25  1.5  1.75  2.0   3.0 -0.694443  ...   
b      2.0  1.5  0.707107  1.0  1.25  1.5  1.75  2.0   2.0 -0.663642  ...   

                         data2                                          \
           75%       max count      mean       std       min       25%   
key1                                                                     
a     0.146960  0.984036   3.0 -1.298603  1.049520 -2.102542 -1.892256   
b     0.007548  0.678737   2.0  0.792094  1.803057 -0.482860  0.154617   

                                    
           50%       75%       max  
key1                                
a    -1.681970 -0.896633 -0.111297  
b     0.792094  1.429571  2.067048  

[2 rows x 24 columns]

In [39]:
tips = pd.read_csv("https://bit.ly/3VyE0vP")
tips

,total_bill,tip,smoker,day,time,size
0,16.99,1.01,No,Sun,Dinner,2
1,10.34,1.66,No,Sun,Dinner,3
2,21.01,3.50,No,Sun,Dinner,3
3,23.68,3.31,No,Sun,Dinner,2
4,24.59,3.61,No,Sun,Dinner,4
...,...,...,...,...,...,...
239,29.03,5.92,No,Sat,Dinner,3
240,27.18,2.00,Yes,Sat,Dinner,2
241,22.67,2.00,Yes,Sat,Dinner,2
242,17.82,1.75,No,Sat,Dinner,2


In [40]:
tips["tip_pct"] = tips["tip"] / tips['total_bill']
tips

,total_bill,tip,smoker,day,time,size,tip_pct
0,16.99,1.01,No,Sun,Dinner,2,0.059447
1,10.34,1.66,No,Sun,Dinner,3,0.160542
2,21.01,3.50,No,Sun,Dinner,3,0.166587
3,23.68,3.31,No,Sun,Dinner,2,0.139780
4,24.59,3.61,No,Sun,Dinner,4,0.146808
...,...,...,...,...,...,...,...
239,29.03,5.92,No,Sat,Dinner,3,0.203927
240,27.18,2.00,Yes,Sat,Dinner,2,0.073584
241,22.67,2.00,Yes,Sat,Dinner,2,0.088222
242,17.82,1.75,No,Sat,Dinner,2,0.098204


In [44]:
grouped=tips.groupby(['day','smoker'])
grouped_pct=grouped['tip_pct']
grouped_pct.agg('mean')

day   smoker
Fri   No        0.151650
      Yes       0.174783
Sat   No        0.158048
      Yes       0.147906
Sun   No        0.160113
      Yes       0.187250
Thur  No        0.160298
      Yes       0.163863
Name: tip_pct, dtype: float64

In [47]:
grouped_pct.agg(['mean','std',ran])

mean       std       ran
day  smoker                              
Fri  No      0.151650  0.028123  0.067349
     Yes     0.174783  0.051293  0.159925
Sat  No      0.158048  0.039767  0.235193
     Yes     0.147906  0.061375  0.290095
Sun  No      0.160113  0.042347  0.193226
     Yes     0.187250  0.154134  0.644685
Thur No      0.160298  0.038774  0.193350
     Yes     0.163863  0.039389  0.151240

In [48]:
grouped_pct.agg(m='mean',s='std',peak_2_peak=ran)

m         s  peak_2_peak
day  smoker                                 
Fri  No      0.151650  0.028123     0.067349
     Yes     0.174783  0.051293     0.159925
Sat  No      0.158048  0.039767     0.235193
     Yes     0.147906  0.061375     0.290095
Sun  No      0.160113  0.042347     0.193226
     Yes     0.187250  0.154134     0.644685
Thur No      0.160298  0.038774     0.193350
     Yes     0.163863  0.039389     0.151240

In [49]:
# apply several functions
fun=['count','mean','min','max','median']
summa=grouped[['tip_pct','total_bill']].agg(fun)
summa

tip_pct                                         total_bill  \
              count      mean       min       max    median      count   
day  smoker                                                              
Fri  No           4  0.151650  0.120385  0.187735  0.149241          4   
     Yes         15  0.174783  0.103555  0.263480  0.173913         15   
Sat  No          45  0.158048  0.056797  0.291990  0.150152         45   
     Yes         42  0.147906  0.035638  0.325733  0.153624         42   
Sun  No          57  0.160113  0.059447  0.252672  0.161665         57   
     Yes         19  0.187250  0.065660  0.710345  0.138122         19   
Thur No          45  0.160298  0.072961  0.266312  0.153492         45   
     Yes         17  0.163863  0.090014  0.241255  0.153846         17   

                                              
                  mean    min    max  median  
day  smoker                                   
Fri  No      18.420000  12.46  22.75  19.235  
     Yes     16.813333   5.75  40.17  13.420  
Sat  No      19.661778   7.25  48.33  17.820  
     Yes     21.276667   3.07  50.81  20.390  
Sun  No      20.506667   8.77  48.17  18.430  
     Yes     24.120000   7.25  45.35  23.100  
Thur No      17.113111   7.51  41.19  15.950  
     Yes     19.190588  10.34  43.11  16.470

In [53]:
# apply and rename different function
fun=[('m1','mean'),('m2','min')]
summa=grouped[['tip_pct','total_bill']].agg(fun)
summa

tip_pct           total_bill       
                   m1        m2         m1     m2
day  smoker                                      
Fri  No      0.151650  0.120385  18.420000  12.46
     Yes     0.174783  0.103555  16.813333   5.75
Sat  No      0.158048  0.056797  19.661778   7.25
     Yes     0.147906  0.035638  21.276667   3.07
Sun  No      0.160113  0.059447  20.506667   8.77
     Yes     0.187250  0.065660  24.120000   7.25
Thur No      0.160298  0.072961  17.113111   7.51
     Yes     0.163863  0.090014  19.190588  10.34

In [54]:
# different columns apply different function
grouped.agg({'tip':'max','size':'sum'})

tip  size
day  smoker             
Fri  No       3.50     9
     Yes      4.73    31
Sat  No       9.00   115
     Yes     10.00   104
Sun  No       6.00   167
     Yes      6.50    49
Thur No       6.70   112
     Yes      5.00    40

In [58]:
# different columns apply several different functions
grouped.agg({'tip':['min','max'],'size':['sum','mean']})

tip        size          
              min    max  sum      mean
day  smoker                            
Fri  No      1.50   3.50    9  2.250000
     Yes     1.00   4.73   31  2.066667
Sat  No      1.00   9.00  115  2.555556
     Yes     1.00  10.00  104  2.476190
Sun  No      1.01   6.00  167  2.929825
     Yes     1.50   6.50   49  2.578947
Thur No      1.25   6.70  112  2.488889
     Yes     2.00   5.00   40  2.352941

In [59]:
# code continuation
(
    grouped[["tip_pct", "total_bill"]]
    .agg(["count", "mean", "median"])
    .rename_axis(index=["day", "smoker"])
)

tip_pct                     total_bill                   
              count      mean    median      count       mean  median
day  smoker                                                          
Fri  No           4  0.151650  0.149241          4  18.420000  19.235
     Yes         15  0.174783  0.173913         15  16.813333  13.420
Sat  No          45  0.158048  0.150152         45  19.661778  17.820
     Yes         42  0.147906  0.153624         42  21.276667  20.390
Sun  No          57  0.160113  0.161665         57  20.506667  18.430
     Yes         19  0.187250  0.138122         19  24.120000  23.100
Thur No          45  0.160298  0.153492         45  17.113111  15.950
     Yes         17  0.163863  0.153846         17  19.190588  16.470

In [60]:
# Disable index
tips.groupby(["day", "smoker"], as_index=False)[['total_bill', 'tip', 'size', 'tip_pct']].mean()

,day,smoker,total_bill,tip,size,tip_pct
0,Fri,No,18.420000,2.812500,2.250000,0.151650
1,Fri,Yes,16.813333,2.714000,2.066667,0.174783
2,Sat,No,19.661778,3.102889,2.555556,0.158048
3,Sat,Yes,21.276667,2.875476,2.476190,0.147906
4,Sun,No,20.506667,3.167895,2.929825,0.160113
5,Sun,Yes,24.120000,3.516842,2.578947,0.187250
6,Thur,No,17.113111,2.673778,2.488889,0.160298
7,Thur,Yes,19.190588,3.030000,2.352941,0.163863


# split-apply-combine

## top

In [63]:
def top(df,n=5,column='tip_pct'):
    return df.sort_values(column,ascending=False)[:n]

top(tips,n=6)

,total_bill,tip,smoker,day,time,size,tip_pct
172,7.25,5.15,Yes,Sun,Dinner,2,0.710345
178,9.60,4.00,Yes,Sun,Dinner,2,0.416667
67,3.07,1.00,Yes,Sat,Dinner,1,0.325733
232,11.61,3.39,No,Sat,Dinner,2,0.291990
183,23.17,6.50,Yes,Sun,Dinner,4,0.280535
109,14.31,4.00,Yes,Sat,Dinner,2,0.279525


In [64]:
tips.sort_values('tip_pct',ascending=False).head()

,total_bill,tip,smoker,day,time,size,tip_pct
172,7.25,5.15,Yes,Sun,Dinner,2,0.710345
178,9.60,4.00,Yes,Sun,Dinner,2,0.416667
67,3.07,1.00,Yes,Sat,Dinner,1,0.325733
232,11.61,3.39,No,Sat,Dinner,2,0.291990
183,23.17,6.50,Yes,Sun,Dinner,4,0.280535


function > direct chaining

In [65]:
import timeit
time_fun=timeit.timeit(
    stmt='top(tips,n=6)',
    globals=globals(),
    number=1000)
time_direct=timeit.timeit(
    stmt='tips.sort_values("tip_pct",ascending=False).head(6)',
    globals=globals(),
    number=1000)
print(f'function version: {time_fun:.6f} seconds')
print(f'direct chaining: {time_direct:.6f} second')

function version: 0.189072 seconds
direct chaining: 0.182914 second


In [66]:
tips.groupby('smoker')[['total_bill','tip','size','tip_pct']].apply(top)

total_bill   tip  size   tip_pct
smoker                                      
No     232       11.61  3.39     2  0.291990
       149        7.51  2.00     2  0.266312
       51        10.29  2.60     2  0.252672
       185       20.69  5.00     5  0.241663
       88        24.71  5.85     2  0.236746
Yes    172        7.25  5.15     2  0.710345
       178        9.60  4.00     2  0.416667
       67         3.07  1.00     1  0.325733
       183       23.17  6.50     4  0.280535
       109       14.31  4.00     2  0.279525

In [67]:
tips.groupby('smoker')[['total_bill','tip','size','tip_pct']].apply(top,n=3)

total_bill   tip  size   tip_pct
smoker                                      
No     232       11.61  3.39     2  0.291990
       149        7.51  2.00     2  0.266312
       51        10.29  2.60     2  0.252672
Yes    172        7.25  5.15     2  0.710345
       178        9.60  4.00     2  0.416667
       67         3.07  1.00     1  0.325733

In [69]:
summa.unstack('smoker')

tip_pct                               total_bill                    \
              m1                  m2                   m1                m2   
smoker        No       Yes        No       Yes         No        Yes     No   
day                                                                           
Fri     0.151650  0.174783  0.120385  0.103555  18.420000  16.813333  12.46   
Sat     0.158048  0.147906  0.056797  0.035638  19.661778  21.276667   7.25   
Sun     0.160113  0.187250  0.059447  0.065660  20.506667  24.120000   8.77   
Thur    0.160298  0.163863  0.072961  0.090014  17.113111  19.190588   7.51   

               
               
smoker    Yes  
day            
Fri      5.75  
Sat      3.07  
Sun      7.25  
Thur    10.34

## fillna

In [70]:
rng=np.random.default_rng(2321)
s=pd.Series(rng.standard_normal(6),index=['a','b','c','d','e','f'])
group=['east','west','east','east','west','west']
s

a   -0.540487
b    0.053875
c    1.327179
d    1.525294
e    0.786086
f    0.960740
dtype: float64

In [73]:
s[::2]=np.nan

In [79]:
s=s.fillna(s.mean())
s

a    0.846636
b    0.053875
c    0.846636
d    1.525294
e    0.846636
f    0.960740
dtype: float64

In [80]:
s[['a','f']]=np.nan
s

a         NaN
b    0.053875
c    0.846636
d    1.525294
e    0.846636
f         NaN
dtype: float64

In [81]:
s.groupby(group).size()

east    3
west    3
dtype: int64

In [82]:
s.groupby(group).count()

east    2
west    2
dtype: int64

In [84]:
s=s.fillna(s.groupby(group).transform('mean'))
s

a    1.185965
b    0.053875
c    0.846636
d    1.525294
e    0.846636
f    0.450256
dtype: float64

In [93]:
s.iloc[::3]=np.nan

In [89]:
fill={'east':0.5,'west':-1}
def fill_fun(group):
    return group.fillna(fill[group.name])
s.groupby(group).apply(fill_fun)

east  a    0.500000
      c    0.846636
      d    0.500000
west  b    0.053875
      e    0.846636
      f    0.450256
dtype: float64

In [94]:
def fill_mean(group):
    return group.fillna(group.mean())
s.groupby(group).apply(fill_mean)

east  a    0.846636
      c    0.846636
      d    0.846636
west  b    0.053875
      e    0.846636
      f    0.450256
dtype: float64

## normalization

In [4]:
df=pd.DataFrame({'key':['a','b','c']*3,'value':np.arange(9)})
df

,key,value
0,a,0
1,b,1
2,c,2
3,a,3
4,b,4
5,c,5
6,a,6
7,b,7
8,c,8


In [5]:
group=df.groupby('key')
group.mean()

,value
key,
a,3.0
b,4.0
c,5.0


In [6]:
group.transform('mean')

,value
0,3.0
1,4.0
2,5.0
3,3.0
4,4.0
5,5.0
6,3.0
7,4.0
8,5.0


In [7]:
group.apply('mean')

,value
key,
a,3.0
b,4.0
c,5.0


In [8]:
def normalization(x):
    return (x-x.mean())/x.std()
group.apply(normalization)

value
key         
a   0   -1.0
    3    0.0
    6    1.0
b   1   -1.0
    4    0.0
    7    1.0
c   2   -1.0
    5    0.0
    8    1.0

In [9]:
group.transform(normalization)

,value
0,-1.0
1,-1.0
2,-1.0
3,0.0
4,0.0
5,0.0
6,1.0
7,1.0
8,1.0


In [12]:
n=(df['value']-group.transform('mean')['value'])/group.transform('std')['value']
n

0   -1.0
1   -1.0
2   -1.0
3    0.0
4    0.0
5    0.0
6    1.0
7    1.0
8    1.0
Name: value, dtype: float64

## pivot table and cross table

In [126]:
tips

,total_bill,tip,smoker,day,time,size,tip_pct
0,16.99,1.01,No,Sun,Dinner,2,0.059447
1,10.34,1.66,No,Sun,Dinner,3,0.160542
2,21.01,3.50,No,Sun,Dinner,3,0.166587
3,23.68,3.31,No,Sun,Dinner,2,0.139780
4,24.59,3.61,No,Sun,Dinner,4,0.146808
...,...,...,...,...,...,...,...
239,29.03,5.92,No,Sat,Dinner,3,0.203927
240,27.18,2.00,Yes,Sat,Dinner,2,0.073584
241,22.67,2.00,Yes,Sat,Dinner,2,0.088222
242,17.82,1.75,No,Sat,Dinner,2,0.098204


In [131]:
tips.pivot_table(index=['time','smoker'],columns='day',
                 values='tip_pct',aggfunc='mean',margins=True,fill_value=-999)

day                 Fri         Sat         Sun        Thur       All
time   smoker                                                        
Dinner No      0.139622    0.158048    0.160113    0.159744  0.158653
       Yes     0.165347    0.147906    0.187250 -999.000000  0.160828
Lunch  No      0.187735 -999.000000 -999.000000    0.160311  0.160920
       Yes     0.188937 -999.000000 -999.000000    0.163863  0.170404
All            0.169913    0.153152    0.166897    0.161276  0.160803

In [135]:
pd.crosstab([tips['time'],tips['smoker']],tips['day'],margins=True)

day            Fri  Sat  Sun  Thur  All
time   smoker                          
Dinner No        3   45   57     1  106
       Yes       9   42   19     0   70
Lunch  No        1    0    0    44   45
       Yes       6    0    0    17   23
All             19   87   76    62  244

**agg()** is using after groupby to aggragation the value.

**transform()** is converting from A to B, just for one line applied.

**apply()** is applying on value, but don't keep the original dataframe.